# Task 3 – Statistical tests
**Who:** Joud & Maram  
**Input:** `food_final.csv`, the final dataset for the model. 

Every test follows **Hypotheses → Assumptions → Test → Result → Interpretation**, with α = 0.05.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

pandas and numpy for the tables, scipy.stats for the tests (as in Lab 06), matplotlib for the plots.

In [ ]:
final = pd.read_csv('food_final.csv')
print(final.shape)

We read the final dataset file. (Write the number of rows and columns here)

## Statistical tests
For the tests that compare **before and after** the change we use only **Pre** (before 1 July 2018) and **Post** (after 30 June 2019), and leave out the transition year, which is a mix of both. The transition year is tested separately in H4.

In [ ]:
pre_post = final[final['Period'].isin(['Pre', 'Post'])]
pre_post['Period'].value_counts()

Number of Pre and Post inspections used in the before/after tests. (Write here the two numbers.)

**Effect size for chi-square tests (Cramér's V).** With more than 100,000 rows almost every difference is significant, so we also report how **big** it is. Cramér's V = √(χ² / (n × (k − 1))), where k is the smaller number of rows or columns of the table. It goes from 0 (no relationship) to 1 (perfect relationship); about 0.1 = small, 0.3 = medium, 0.5 = large. (This formula is new for us; it is the usual effect size for chi-square.)

In [ ]:
def cramers_v(table):
    chi2, p, dof, expected = stats.chi2_contingency(table)
    n = table.values.sum()
    k = min(table.shape) - 1
    return np.sqrt(chi2 / (n * k))

A small function that calculates Cramér's V from a crosstab, so we can use it for every chi-square test.

### H1a – Did the distribution of results change?
**H0:** the shares of Pass / Pass w/ Conditions / Fail are the same before and after the change.  
**H1:** the shares are different.  
**Assumptions:** two categorical variables (Period, Results), independent inspections, every expected count ≥ 5.  
**Test:** chi-square test of independence.

In [ ]:
table_h1a = pd.crosstab(pre_post['Period'], pre_post['Results'])
table_h1a

The counts of each result before and after.

In [ ]:
pd.crosstab(pre_post['Period'], pre_post['Results'], normalize='index').round(3)

The same as shares (each row adds up to 1). (Write here how each share changes from Pre to Post.)

In [ ]:
chi2, p, dof, expected = stats.chi2_contingency(table_h1a)
print(f'Chi-square = {chi2:.2f}, df = {dof}, p-value = {p:.3e}, min expected = {expected.min():.0f}')
print(f"Cramer's V = {cramers_v(table_h1a):.3f}")

**Result and interpretation:** (Write here: reject or fail to reject H0, Cramér's V and its size, and what it means in words.)

### H1b – Did the number of violations change?
**H0:** the number of violations per inspection has the same distribution before and after.  
**H1:** the distributions are different.  
**Assumptions:** two independent groups, a numeric variable. The counts are not normal (right-skewed, many zeros, see the histogram in Task 1), so we use the non-parametric **Mann–Whitney U** test, as in Lab 06 and Assignment 2.

In [ ]:
pre_post.groupby('Period')[['Num_Violations', 'Num_Critical', 'Num_Serious']].median()

Median number of violations before and after. (Write here.)

In [ ]:
pre_post.groupby('Period')[['Num_Violations', 'Num_Critical', 'Num_Serious']].mean().round(2)

Mean number of violations before and after. (Write here.)

In [ ]:
pre_v = pre_post[pre_post['Period'] == 'Pre']['Num_Violations']
post_v = pre_post[pre_post['Period'] == 'Post']['Num_Violations']
u_stat, u_p = stats.mannwhitneyu(pre_v, post_v, alternative='two-sided')
rank_biserial = 2 * (u_stat / (len(pre_v) * len(post_v))) - 1
print(f'Mann-Whitney U = {u_stat:.0f}, p-value = {u_p:.3e}, rank-biserial = {rank_biserial:.3f}')

**Result and interpretation:** (Write here.) The rank-biserial correlation is the effect size (same formula as in Assignment 2): a negative value means Pre inspections tend to have **fewer** violations than Post inspections.

In [ ]:
for col in ['Num_Critical', 'Num_Serious']:
    u, p_col = stats.mannwhitneyu(pre_post[pre_post['Period'] == 'Pre'][col], pre_post[pre_post['Period'] == 'Post'][col], alternative='two-sided')
    print(f'{col}: U = {u:.0f}, p-value = {p_col:.3e}')

The same test for critical and serious violations separately. (Write here. From Task 2 we expect critical to change little and serious to change a lot, because the new Priority Foundation level is much wider than the old Serious level.)

### H1c – Did the mix of violation topics change?
**H0:** the share of inspections with each topic (rodents, hand washing, …) is the same before and after.  
**H1:** the share is different.  
**Test:** one chi-square test per topic (Period × topic yes/no).

In [ ]:
topics = ['has_rodents', 'has_temperature', 'has_handwashing', 'has_toxic', 'has_water_sewage', 'has_dirty_surfaces',
          'has_thermometers', 'has_food_handler', 'has_smoking', 'has_summary_report']
rows = []
for t in topics:
    table = pd.crosstab(pre_post['Period'], pre_post[t])
    chi2_t, p_t, dof_t, exp_t = stats.chi2_contingency(table)
    rows.append({'Topic': t,
                 'Pre share': pre_post[pre_post['Period'] == 'Pre'][t].mean().round(3),
                 'Post share': pre_post[pre_post['Period'] == 'Post'][t].mean().round(3),
                 'Chi-square': round(chi2_t, 1), 'p-value': p_t, "Cramer's V": round(cramers_v(table), 3)})
h1c = pd.DataFrame(rows).sort_values("Cramer's V", ascending=False)
h1c

One row per topic, sorted by effect size (biggest change first).  
**Note on many tests:** we run 10 tests, so with α = 0.05 one of them could be significant just by chance (a Type I error). We therefore focus on the topics with a clear effect size, not only on the p-values.  
**Result and interpretation:** (Write here which topics changed the most.)

### H1 – Conclusion
(Write here one short paragraph that combines H1a, H1b and H1c: for example, the fail rate changed little, but the way results and violations are recorded changed a lot.)

### H4 – Was the 90-day grace period different from the rest of the transition year?
**H0:** the shares of Pass / Pass w/ Conditions / Fail are the same in the grace period (1 July – 30 September 2018) and in the rest of the transition year.  
**H1:** they are different.  
**Test:** chi-square test of independence.

In [ ]:
transition = final[final['Period'].isin(['Grace (90 days)', 'Late transition'])]
table_h4 = pd.crosstab(transition['Period'], transition['Results'])
pd.crosstab(transition['Period'], transition['Results'], normalize='index').round(3)

Shares of each result in the two parts of the transition year. (Write here. In Task 1 we saw about 12% clean Pass in the grace period vs about 24% after.)

In [ ]:
chi2, p, dof, expected = stats.chi2_contingency(table_h4)
print(f'Chi-square = {chi2:.2f}, df = {dof}, p-value = {p:.3e}')
print(f"Cramer's V = {cramers_v(table_h4):.3f}")

**Result and interpretation:** (Write here.)

### H5 – Does a business's previous result help predict its current result?
**H0:** failing the previous inspection and failing the current inspection are not related.  
**H1:** they are related.  
**Assumptions:** we use only **normal visits** (no re-inspections), because a re-inspection right after a fail is usually a pass (Maram's notebook), and only businesses that have a previous visit.  
**Test:** chi-square test of independence.

In [ ]:
normal = final[(final['Is_Reinspection'] == 0) & (final['First_Inspection'] == 0)]
table_h5 = pd.crosstab(normal['Prev_Fail'], normal['Fail'])
pd.crosstab(normal['Prev_Fail'], normal['Fail'], normalize='index').round(3)

Fail rate now, for businesses that passed (0) or failed (1) at their previous visit. (Write here.)

In [ ]:
chi2, p, dof, expected = stats.chi2_contingency(table_h5)
print(f'Chi-square = {chi2:.2f}, df = {dof}, p-value = {p:.3e}')
print(f"Cramer's V = {cramers_v(table_h5):.3f}")

**Result and interpretation:** (Write here. If p < 0.05, the past helps predict the future, which supports using the history columns in the model, even though the city says the data is "not intended to predict future findings".)

### Risk level
The chi-square test for Risk × Fail is in `Extra.ipynb` (χ² = 72.80, p ≈ 1.6 × 10⁻¹⁶, Spearman ρ = 0.013): the fail rate differs between risk levels, but the relationship is very weak, and part of it is explained by the much higher share of License inspections in Risk 3 (37.6% vs 11.2% in Risk 1).

## Summary of the results
(Write here a table: hypothesis, test, statistic, p-value, effect size, decision. This table goes into the brief report.)